In [ ]:
# pip install tensorflow datasets pillow 

In [ ]:
from datasets import load_dataset
import os
import numpy as np
import matplotlib.pyplot as plt
import random
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

In [ ]:
# loading dataset
dataset = load_dataset("mnist")

In [ ]:
# Function to save images into folders
def save_split(split, split_name):
    for i in range(10):
        os.makedirs(f"data/mnist_data/{split_name}/{i}", exist_ok=True)

    for idx, example in enumerate(split):
        image = example["image"]  # PIL image
        label = example["label"]
        image.save(f"data/mnist_data/{split_name}/{label}/{idx}.png")

In [ ]:
# saving train & test images
save_split(dataset["train"], "train")
save_split(dataset["test"], "test")

In [ ]:
# displaying sample images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for i in range(10):
    label = str(i)
    folder = f"data/mnist_data/train/{label}"
    img_file = random.choice(os.listdir(folder))
    
    img = Image.open(os.path.join(folder, img_file))
    
    ax = axes[i // 5, i % 5]
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Label: {label}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# loading data using keras
img_size = 28
batch_size = 32

train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    "data/mnist_data/train",
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_directory(
    "data/mnist_data/test",
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode="categorical"
)

In [ ]:
# defining CNN model
model = models.Sequential([
    layers.Input(shape=(28,28,1)),
    # block 1
    layers.Conv2D(16, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),
    
    # block 2
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),

    # block 3
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.summary()

In [ ]:
# compiling the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# training the model
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=test_generator
)

In [ ]:
# model evaluation
loss, acc = model.evaluate(test_generator)
print(f"Test Accuracy: {acc:.2f}")

In [ ]:
# making predictions
import numpy as np

# getting one batch
images, labels = next(test_generator)

predictions = model.predict(images)

# showing predictions
fig, axes = plt.subplots(2, 5, figsize=(10,5))

for i in range(10):
    ax = axes[i//5, i%5]
    ax.imshow(images[i].squeeze(), cmap='gray')
    
    pred_label = np.argmax(predictions[i])
    true_label = np.argmax(labels[i])
    
    ax.set_title(f"Pred:{pred_label} True:{true_label}")
    ax.axis("off")

plt.show()

**Exercise** 
1. Create 10 digit images (0–9). Draw them yourself and save as PNG/JPG. Preprocess each image accordingly and run predictions using your trained model. What is the model's performance?


In [ ]:
# Preprocess each image
def preprocess_image(image_path):
    img = Image.open(image_path).convert('L')  # convert to grayscale
    img = img.resize((28, 28))                  # resize to 28x28
    img = np.array(img)
    img = 255 - img                             # invert (MNIST is white digit on black)
    img = img / 255.0                           # normalize
    img = img.reshape(1, 28, 28, 1)            # reshape for model
    return img

# Predict all digits
image_folder = 'my_images'
true_labels = list(range(10))  # 0-9
predicted_labels = []

plt.figure(figsize=(15, 5))

for i in range(10):
    image_path = os.path.join(image_folder, f'digit_{i}.png')
    img = preprocess_image(image_path)
    prediction = model.predict(img, verbose=0)
    predicted_label = np.argmax(prediction)
    predicted_labels.append(predicted_label)

    plt.subplot(2, 5, i+1)
    plt.imshow(img.reshape(28, 28), cmap='gray')
    plt.title(f'True: {i}\nPred: {predicted_label}')
    plt.axis('off')

plt.suptitle('My Handdrawn Digits - Predictions')
plt.tight_layout()
plt.show()

# Performance
correct = sum(p == t for p, t in zip(predicted_labels, true_labels))
accuracy = correct / 10
print(f"\nPredictions: {predicted_labels}")
print(f"True labels: {true_labels}")
print(f"Accuracy: {accuracy * 100:.1f}% ({correct}/10 correct)")

2. Using the Fashion MNIST dataset, load and preprocess the data. Use the same CNN architecture from the MNIST exercise. Train the model for 5 epochs. Report model performance. (Use accuracy and F1 score)

In [ ]:
import tensorflow as tf
from sklearn.metrics import f1_score, classification_report

# Load and preprocess data
(X_train_fashion, y_train_fashion), (X_test_fashion, y_test_fashion) = tf.keras.datasets.fashion_mnist.load_data()

X_train = X_train_fashion.reshape(-1, 28, 28, 1) / 255.0
X_test  = X_test_fashion.reshape(-1, 28, 28, 1)  / 255.0

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Define Model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# 3. Train for 5 epochs
history = model.fit(X_train, y_train_fashion,
                    epochs=5,
                    validation_data=(X_test, y_test_fashion))

# 4. Evaluate — Accuracy
test_loss, test_accuracy = model.evaluate(X_test, y_test_fashion, verbose=0)
print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")

# 5. F1 Score
y_pred = np.argmax(model.predict(X_test), axis=1)
f1 = f1_score(y_test_fashion, y_pred, average='weighted')
print(f"Weighted F1 Score: {f1:.4f}")

# Detailed report per class
print("\nClassification Report:")
print(classification_report(y_test_fashion, y_pred, target_names=class_names))

# 6. Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()